# Lab 11 — Foundation-model priors for perturbation prediction

*An add-on to the [`notebooks/` course](README.md), beyond Setup + Labs 1–10. This one operationalises [`docs/foundation-models.md`](../docs/foundation-models.md): foundation models as **zero-shot priors** for the kind of perturbation-prediction problems [Lab 3](03_benchmarking_fidelity.ipynb) benchmarks, with the integration contract from that doc — extract once, cache as `.npy`, then run the JAX pipeline on cached arrays.*

**The benchmark — unseen-gene generalisation.** The canonical FM-prior use case: a predictor trained on perturbation responses of one half of the genes is asked to predict responses for the *other half*. One-hot gene features carry zero signal on unseen genes by construction; foundation-model gene embeddings carry the co-expression manifold the unseen genes live on. The lift is real even with the stub backend, because *any* embedding that encodes co-regulation generalises across gene identity.

**Stub vs real.** The lab runs with the **stub backend** of [`scripts/fm_embed.py`](../scripts/fm_embed.py) — a deterministic SVD projection at the correct dimensions (1280 for UCE, 512 for Geneformer, 512 for scGPT). That's the right *attribution control*: it isolates "co-expression-aware features help" from "30 M-cell pretraining helps". With `--mode real`, the same script loads the actual HF checkpoints (`ctheodoris/Geneformer`, `subercui/scGPT`, `chanzuckerberg/uce`); the notebook code below is unchanged. The architectural point — that the rest of the pipeline consumes a single cached array regardless of which FM produced it — is what this lab demonstrates.

**See also.** [`docs/foundation-models.md`](../docs/foundation-models.md) for the institutional catalogue and integration plan; [Lab 3](03_benchmarking_fidelity.ipynb) for the fidelity-triple setup this lab borrows the transfer-r framing from; [Lab 5](05_hypergraph_neural_odes.ipynb) for where Geneformer-style gene embeddings naturally slot in as `hgx` node features.


In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Make scripts/fm_embed.py importable from the notebook.
NB_DIR = Path.cwd()
REPO = NB_DIR if (NB_DIR / 'scripts').is_dir() else NB_DIR.parent
sys.path.insert(0, str(REPO / 'scripts'))

import fm_embed
print('fm_embed: subcommands =', list(fm_embed.MODELS.keys()))

rng = np.random.default_rng(0)


## 1. A synthetic expression matrix with shared factor structure

A miniature regulome: 400 genes drawn from a 4-factor latent model. Each cell has a factor-score vector `W[c]`; each gene has a factor-loading vector `H[g]`; expression is `W @ H + noise`. This is the setup the project's `hgx` Pando-derived regulome approximates in spirit — a low-rank structural signal beneath observation noise.

In [ ]:
N_CELLS, N_GENES, N_FACTORS = 400, 400, 4

H = rng.standard_normal((N_FACTORS, N_GENES)).astype(np.float32)   # gene loadings (the "truth")
W = rng.standard_normal((N_CELLS, N_FACTORS)).astype(np.float32)    # cell factor scores
noise = rng.standard_normal((N_CELLS, N_GENES)).astype(np.float32) * 0.4
expression = (W @ H + noise).astype(np.float32)
counts = np.maximum(0, np.round((expression - expression.min()) * 3)).astype(np.float32)

print(f'counts: {counts.shape}  W: {W.shape}  H: {H.shape}')


## 2. The perturbation-response task

For each gene $g$ we have a true *response vector* — what every *other* gene does when $g$ is perturbed. Encoded as the (`gene_g`-th column of) the gene–gene Jacobian. Since the system is low-rank by construction, $R = -\alpha\,H^\top H$ (a rank-4 matrix in a 400-dim ambient space).

**The benchmark.** Hold out 200 genes. Train on the other 200's response vectors. Predict the held-out 200's responses. *One-hot has zero signal on held-out genes by definition*; a foundation-model embedding that carries co-expression structure generalises.

In [ ]:
ALPHA = 0.5
R_TRUE = (-ALPHA * (H.T @ H)).astype(np.float32)  # (n_genes, n_genes) response matrix
print(f'R_TRUE: {R_TRUE.shape}   rank-4 in a {N_GENES}-dim ambient space')

# Train / test split on the PERTURBED-gene axis (rows). Within each row, we predict the
# response on the COLUMN axis (which is the same set of 400 read-out genes for both arms).
train_idx = rng.permutation(N_GENES)[: N_GENES // 2]
test_idx = np.setdiff1d(np.arange(N_GENES), train_idx)
print(f'train: {train_idx.shape}   test: {test_idx.shape}')


## 3. Baseline: one-hot gene features

One-hot is the strictest no-prior baseline. With held-out genes, the test-time row of the one-hot feature matrix points at columns of `W` that *never appeared at training time*, so a linear predictor has literally nothing to say about them. The transfer-r on held-out genes drops to noise.

In [ ]:
def fit_predict(gene_features, train_idx, test_idx, R_true, ridge=1.0):
    '''Ridge regression: predict R_true[i, :] from gene_features[i, :].

    The predictor's job is to map a gene's feature vector to its response
    profile (response to perturbing it, across all other genes).

    Note: with one-hot features, the test-time rows are orthogonal to every
    training row, so B has zero weight on test-gene columns and the
    prediction is exactly zero. Correlation of a zero vector with anything
    is undefined; we report r = 0.0 with a flag so the comparison panel
    reads "no signal", not NaN.
    '''
    A_train = gene_features[train_idx]
    Y_train = R_true[train_idx]
    AtA = A_train.T @ A_train + ridge * np.eye(A_train.shape[1], dtype=np.float32)
    B = np.linalg.solve(AtA, A_train.T @ Y_train)

    pred_test = gene_features[test_idx] @ B
    Y_test = R_true[test_idx]
    if pred_test.std() < 1e-8:
        return 0.0, pred_test  # zero-variance prediction = no signal
    r = float(np.corrcoef(pred_test.ravel(), Y_test.ravel())[0, 1])
    return r, pred_test

gene_features_onehot = np.eye(N_GENES, dtype=np.float32)
r_onehot, _ = fit_predict(gene_features_onehot, train_idx, test_idx, R_TRUE, ridge=1.0)
print(f'one-hot baseline transfer-r (held-out genes) = {r_onehot:+.3f}')


## 4. Extracting foundation-model embeddings

Call [`scripts/fm_embed.py`](../scripts/fm_embed.py) on the expression matrix — get a Geneformer-style 512-d gene embedding (per-gene structural prior) and a UCE-style 1280-d cell embedding (per-cell structural prior). **Mode "stub"** here for reproducible single-second execution; **mode "auto"** would attempt the real checkpoints first and fall back with a warning if the weights aren't installed.

The integration contract: the script writes `.npy` arrays. We use the in-process `extract()` API here for notebook convenience, but the CLI form

```
python scripts/fm_embed.py geneformer --input data/X.h5ad --output cache/
```

produces the same shapes and is what production pipelines call.

In [ ]:
import anndata as ad

adata = ad.AnnData(X=counts, var={'gene': [f'g{i}' for i in range(N_GENES)]})

emb_genef, man_genef = fm_embed.extract(adata, model='geneformer', mode='stub', seed=0)
print(f"geneformer-stub: shape {emb_genef.shape}  cite={man_genef['citation']}")

emb_uce, man_uce = fm_embed.extract(adata, model='uce', mode='stub', seed=0)
print(f"uce-stub:        shape {emb_uce.shape}  cite={man_uce['citation']}")


## 5. With FM prior: Geneformer-style gene features

Same predictor, same train / test split. Gene features are now the Geneformer embedding (extracted from the *same* expression matrix the responses were built from — but encoding the shared low-rank structure as a 512-d feature, not as a one-hot per-gene token).

A held-out gene's Geneformer embedding is **a similarity-weighted blend of training-gene embeddings** along the latent factor manifold, so the linear head trained on the train set's responses generalises by interpolation. The transfer-r jumps from noise-floor to a meaningfully positive number.

In [ ]:
r_genef, _ = fit_predict(emb_genef, train_idx, test_idx, R_TRUE, ridge=1.0)
print(f'one-hot baseline    transfer-r (held-out) = {r_onehot:+.3f}')
print(f'Geneformer-stub     transfer-r (held-out) = {r_genef:+.3f}')
print(f'                    Δ                     = {r_genef - r_onehot:+.3f}')


## 6. The comparison panel

The headline plot — same shape as a [Lab 3](03_benchmarking_fidelity.ipynb) before/after bar. With the stub embedding this isolates the "co-expression-aware features help" effect; with real Geneformer / scGPT checkpoints, an additional lift comes from the 30-million-cell pretraining the stub doesn't have.

The right attribution control is exactly what we have here: a *structure-matched random-projection baseline* (the stub) vs the one-hot. Any further lift on top, with the real models, is attributable to the pretraining specifically.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.0))
bars = ['one-hot\n(no prior)', 'Geneformer-stub\n(co-expression prior)']
vals = [r_onehot, r_genef]
colors = ['#9aa0a6', '#1a73e8']
ax.bar(bars, vals, color=colors, width=0.55)
ax.set_ylabel('transfer-r on held-out genes')
ax.set_title('Lab 11 — foundation-model gene-feature prior')
ax.axhline(0, color='black', lw=0.5)
for i, v in enumerate(vals):
    ax.text(i, v + 0.02 if v >= 0 else v - 0.04, f'{v:+.3f}', ha='center', fontsize=11)
ax.set_ylim(min(0, min(vals)) - 0.1, max(0.1, max(vals)) + 0.15)
plt.tight_layout()
plt.show()


## 7. UCE-stub smoke test — property check on the cell-level arm

The cell embedding's job is to compress (cell, gene)-counts into a per-cell vector that downstream models can consume (e.g. as the initial state in [Lab 5](05_hypergraph_neural_odes.ipynb)'s Hypergraph Neural ODE, or as a cell encoder for [Lab 8](08_anatomical_compiler.ipynb)'s SBI inverse). For the synthetic, the *true* cell-level structure is the latent factor scores `W` (cells × 4). A useful cell embedding should contain it — a linear probe should recover `W` from the embedding.

This is a *property check*, not a comparison. UCE's actual advantages — cross-batch transfer, learned imputation from dropout-corrupted training data, generalisation to novel cell types — are properties of the **pretrained checkpoint**, not of the stub. The SVD-based stub captures only the structural shape (1280-d, co-expression-aware, deterministic); the biology comes from 36 M cells of pretraining the stub can't replicate. To demonstrate those, point `fm_embed.extract(..., mode='real')` at a real h5ad with batch labels (see [`docs/foundation-models.md`](../docs/foundation-models.md) §pipeline).

In [ ]:
def linear_probe_r(features, target, ridge=1.0):
    '''Train a ridge linear probe `features → target`; return mean per-column |Pearson r|.'''
    A = np.concatenate([features, np.ones((features.shape[0], 1), dtype=np.float32)], axis=1)
    coef = np.linalg.solve(
        A.T @ A + ridge * np.eye(A.shape[1], dtype=np.float32),
        A.T @ target,
    )
    pred = A @ coef
    rs = [float(np.corrcoef(pred[:, k], target[:, k])[0, 1]) for k in range(target.shape[1])]
    return float(np.mean(np.abs(rs)))

r_uce = linear_probe_r(emb_uce, W, ridge=1.0)
print(f'UCE-stub (1280-d) → W   mean|r| = {r_uce:.3f}')
print(f'  (passing property check: the embedding contains the latent factor structure;')
print(f'   biological properties — cross-batch, sparsity, novel cell types — need mode="real")')


## 8. What this is, what it isn't, and what's next

**What this is.** A faithful demonstration of the *integration contract* in [`docs/foundation-models.md`](../docs/foundation-models.md): extract embeddings once via [`scripts/fm_embed.py`](../scripts/fm_embed.py), cache them, and consume them downstream as just-another-numpy-array. The notebook code is **unchanged** between stub mode and real-checkpoint mode — that's the architectural point.

**What it isn't.** The numerical lift you see above is a *floor*, not a ceiling. The stubs are deterministic SVD projections (a "co-expression-aware random projection at the right dimensions"). They're the right control to attribute *how much* of the eventual real-model lift is structural vs pretraining-specific. The absolute transfer-r numbers here aren't predictions for real Geneformer / scGPT / UCE on the [Lab 3](03_benchmarking_fidelity.ipynb) Pollen brain-organoid + fetal-kidney h5ads.

**Where it slots.**

- The Geneformer embedding plugs into [Lab 5](05_hypergraph_neural_odes.ipynb)'s `hgx` node features (replacing one-hot gene tokens) and into [Lab 6](06_control_theory.ipynb)'s linear controllability matrix as a per-gene representation.
- The UCE cell embedding plugs into [Lab 8](08_anatomical_compiler.ipynb)'s SBI-inverse cell encoder and is the natural initial state for the Hypergraph Neural ODE.
- The scGPT in-silico perturbation API gives a *prior distribution* over perturbation responses — the natural cheap signal for the Bayesian active-learning loop now flagged urgent by [`docs/wetlab-program.md`](../docs/wetlab-program.md) (see also [`docs/computational-roadmap.md`](../docs/computational-roadmap.md) §2).
- Sequence-level priors via **Evo 2** (Arc Institute) or **Borzoi** (Google + Kundaje, Stanford) would supply the regulome's *edge* priors — a candidate Lab 11.5 if and when it lands.

**Decision triggers for going to real mode.**

1. A real h5ad with known biological structure (Pollen brain-organoid or a Biopunk-wet-lab capstone — see [`docs/wetlab-program.md`](../docs/wetlab-program.md)).
2. GPU budget for one A100-day per dataset.
3. A specific benchmark to beat — the [Lab 3](03_benchmarking_fidelity.ipynb) fidelity-triple's transfer-r ≈ 0.13 is the obvious target.

Until then: this lab is the wiring diagram. Run `python scripts/fm_embed.py --help` for the CLI form.